# NIST P-256 시드 재현

lrand48로 공개 곡선 시드를 재현하고 표준 계수의 합동식과 후보 초기화 시각을 확인합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/nist_p256_seed.ipynb)

Colab 런타임은 **CPU**를 선택하고 위에서부터 실행하세요. 첫 코드 셀의 `QUICK = True`는 축소 조건을 사용합니다.

In [ ]:
QUICK = False

import hashlib
import math
import textwrap
import time
from datetime import datetime, timezone
from fractions import Fraction


MOD = 1 << 48
MASK = MOD - 1
A = 0x5DEECE66D
C = 0xB
SUFFIX = 0x330E
A_INV = pow(A, -1, MOD)
SEED_BYTES = 20
BYTE_SHIFT = 40

# FIPS PUB 186-4 부록 D.1.2의 소수체 곡선별 Seed, P-256의 p와 b.
FIPS = {
    "P-192": bytes.fromhex("3045AE6F C8422F64 ED579528 D38120EA E12196D5"),
    "P-224": bytes.fromhex("BD713447 99D5C7FC DC45B59F A3B9AB8F 6A948BC5"),
    "P-256": bytes.fromhex("C49D3608 86E70493 6A6678E1 139D26B7 819F7E90"),
    "P-384": bytes.fromhex("A335926A A319A27A 1D00896A 6773A482 7ACDAC73"),
    "P-521": bytes.fromhex("D09E8800 291CB853 96CC6717 393284AA A0DA64BA"),
}
P = (1 << 256) - (1 << 224) + (1 << 192) + (1 << 96) - 1
B = int("5AC635D8AA3A93E7B3EBBD55769886BC"
        "651D06B0CC53B0F63BCE3C3E27D2604B", 16)
P256_DATE = datetime(1997, 8, 16, 1, 52, 34, tzinfo=timezone.utc)
P224_DATE = datetime(1998, 4, 3, 12, 52, 7, tzinfo=timezone.utc)
P256_INDEX = 1842
P224_INDEX = 7026
TITLES = (
    "lrand48 구현 검증", "P-256 시드와 계수 검증", "P-224 시드 재현",
    "32비트 단어의 상위 비트", "P-256 초기 상태 역추적", "LLL로 공개 시드 역추적",
)


def srand48(t):
    return ((t & 0xFFFFFFFF) << 16) | SUFFIX


def step(state):
    return (A * state + C) & MASK


def back(state):
    return (A_INV * (state - C)) & MASK


def lrand48(state):
    new_state = step(state)
    return new_state, new_state >> 17


def utc_string(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def heading(number):
    print(f"\n## 실험{number} {TITLES[number - 1]}")


def finish(status, **data):
    print(f"판정  {status}")
    return {"status": status, **data}


def verdict(ok):
    return "PASS" if ok else "FAIL"

### 그래프 언어

아래 셀에서 `CHART_LANG`의 기본값을 고르세요. `ko`는 한국어, `en`은 영어, `both`는 각 그래프를 한국어와 영어 순서로 그립니다. 한글 글꼴은 필요할 때 내려받습니다.

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
requested_language = CHART_LANG
if requested_language not in ('ko', 'en', 'both'):
    raise ValueError("CHART_LANG must be ko, en or both")
if requested_language == 'both':
    CHART_LANG = 'ko'
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()
if requested_language == 'both' and CHART_LANG == 'ko':
    CHART_LANG = 'both'
chart_languages = ('ko', 'en') if CHART_LANG == 'both' else (CHART_LANG,)

### 1. 난수 생성기의 기준값

고정 초기값에서 첫 출력과 상태 전이의 역산이 맞는지 확인합니다.

In [ ]:
def experiment1():
    heading(1)
    initial = state = srand48(0)
    outputs = []
    round_trips = []
    for _ in range(5):
        previous = state
        state, output = lrand48(state)
        outputs.append(output)
        round_trips.append(back(state) == previous)
    for i, value in enumerate(outputs, 1):
        print(f"호출 {i}  {value}")
    print(f"초기 상태  0x{initial:012X}")
    print(f"한 걸음 역산 일치  {sum(round_trips)}/5")
    # 첫 호출의 고정 참조값은 순방향과 역방향의 공통 오류도 검출한다.
    ok = initial == 0x330E and outputs[0] == 366850414 and all(round_trips)
    return finish(verdict(ok), initial_state=initial, outputs=outputs,
                  round_trips=round_trips)


generator_check = experiment1()

### 2. P-256 시드와 곡선 계수

지정한 시각에서 후보를 생성해 공개 시드의 모든 바이트와 비교합니다. 이어 SHA-1으로 계산한 값과 표준 계수의 합동식을 검사합니다.

In [ ]:
def reproduce(curve, date, index):
    t = int(date.timestamp())
    state = srand48(t)
    for _ in range((index - 1) * SEED_BYTES):
        state = step(state)
    candidate = bytearray()
    first_state = None
    for _ in range(SEED_BYTES):
        state, output = lrand48(state)
        if not candidate:
            first_state = state
        candidate.append(output >> 23)
    matched = candidate == FIPS[curve]
    print(f"초기화  {utc_string(t)}")
    print(f"Unix 초 t  {t}")
    print(f"후보 {index:,}  누적 호출 {index * SEED_BYTES:,}회")
    print(f"생성  {candidate.hex().upper()}")
    print(f"표준  {FIPS[curve].hex().upper()}")
    print(f"시드 일치  {verdict(matched)}")
    return dict(curve=curve, unix_time=t, index=index, calls=index * SEED_BYTES,
                first_state=first_state, seed=candidate.hex(), seed_match=matched)


def seed_r(seed, p):
    # FIPS 186-4 D.1.1 / ANSI X9.62의 소수체 절차.
    # 첫 해시의 오른쪽 비트만 남기고 최상위 비트를 0으로 한다.
    bits = p.bit_length()
    blocks_after_first = (bits - 1) // 160
    first_bits = bits - 160 * blocks_after_first
    r = int.from_bytes(hashlib.sha1(seed).digest(), "big")
    r &= (1 << (first_bits - 1)) - 1
    base = int.from_bytes(seed, "big")
    for i in range(1, blocks_after_first + 1):
        value = (base + i) % (1 << (8 * len(seed)))
        block = hashlib.sha1(value.to_bytes(len(seed), "big")).digest()
        r = (r << 160) | int.from_bytes(block, "big")
    return r


def experiment2():
    heading(2)
    data = reproduce("P-256", P256_DATE, P256_INDEX)
    r = seed_r(bytes.fromhex(data["seed"]), P)
    residue = r * B * B % P
    relation = residue == (-27) % P
    print("생성 시드에서 계산한 r (16진수)")
    print("\n".join(textwrap.wrap(f"{r:064X}", 32)))
    print(f"r·b² ≡ −27 (mod p)  {verdict(relation)}")
    return finish(verdict(data["seed_match"] and relation), **data,
                  r=r, residue=residue, relation=relation)


p256 = experiment2()

### 3. P-224 시드

같은 생성기로 P-224의 초기화 시각과 후보 번호를 검증합니다.

In [ ]:
def experiment3():
    heading(3)
    data = reproduce("P-224", P224_DATE, P224_INDEX)
    return finish(verdict(data["seed_match"]), **data)


p224 = experiment3()

### 4. 공개 시드의 최상위 비트

각 시드를 32비트 단어로 나누고 첫 비트가 0인 단어 수를 셉니다. 독립인 공정한 비트를 가정했을 때의 양측 확률을 계산합니다.

In [ ]:
def experiment4():
    heading(4)
    counts = {}
    for curve, seed in FIPS.items():
        # 각 단어를 빅엔디언으로 읽을 때 첫 바이트의 비트 7이 MSB다.
        bits = [byte >> 7 for byte in seed[::4]]
        counts[curve] = bits.count(0)
        print(f"{curve}  {''.join(map(str, bits))}  0인 단어 {counts[curve]}/5")
    zeros = sum(counts.values())
    n = sum(len(seed) // 4 for seed in FIPS.values())
    tail = min(zeros, n - zeros)
    two_sided = min(1.0, 2 * sum(math.comb(n, k) for k in range(tail + 1)) / 2**n)
    print(f"전체  {zeros}/{n}개가 0")
    print("독립이고 공정한 비트를 가정한 확률")
    print(f"지정한 20개가 모두 0  2^-20 = {2**-20:.3g}")
    print(f"전체 {n}개가 모두 0  2^-{n} = {2**-n:.3g}")
    print(f"관측만큼 치우칠 양측 확률  {two_sided:.6f}")
    print("판정 기준  곡선별 집계가 참조 개수와 일치")
    expected = {"P-192": 1, "P-224": 1, "P-256": 2, "P-384": 3, "P-521": 2}
    return finish(verdict(counts == expected), counts=counts, zeros=zeros,
                  words=n, prob_all_20=2**-20, prob_all_words=2**-n,
                  two_sided_p=two_sided)


bit_counts = experiment4()

### 5. P-256 초기 상태 역산

첫 출력 바이트를 생성한 상태에서 초기화 표식까지 거꾸로 이동해 순방향 재현 결과와 대조합니다.

In [ ]:
def first_marker(first_state):
    # 하위 16비트 LCG는 2^16 주기로 모든 값을 한 번씩 지난다.
    state = first_state
    for distance in range(1, (1 << 16) + 1):
        state = back(state)
        if state & 0xFFFF == SUFFIX:
            return distance, state >> 16
    raise ArithmeticError("하위 16비트 한 주기에서 초기화 표식 누락")


def experiment5(reproduction=None):
    heading(5)
    if reproduction is None:
        reproduction = reproduce("P-256", P256_DATE, P256_INDEX)
    first = reproduction["first_state"]
    distance, t = first_marker(first)
    print(f"첫 바이트의 전체 상태  0x{first:012X}")
    print(f"첫 0x330E까지 역산  {distance:,}회")
    print(f"복원한 Unix 초  {t}")
    print(utc_string(t))
    ok = (reproduction["seed_match"] and t == int(P256_DATE.timestamp())
          and distance == (P256_INDEX - 1) * SEED_BYTES + 1)
    return finish(verdict(ok), first_state=first, steps=distance, unix_time=t)


reverse_check = experiment5(p256)
lll_result = {'curves': {}}  # 다음 선택 셀을 건너뛰어도 그래프를 그릴 수 있다.

### 6. 선택 실험 · LLL 격자 기저 축소

공개 시드의 상위 바이트로 상태 후보를 찾고 전체 바이트를 검산합니다. 분수 연산에 시간이 걸리며 기본 계산 기한은 8분입니다. `QUICK = True`에서는 생략합니다.

In [ ]:
def check_time(deadline):
    if time.monotonic() >= deadline:
        raise TimeoutError("LLL 실험의 시간 한도에 도달")


def dot(left, right):
    return sum(x * y for x, y in zip(left, right))


def gram_schmidt(basis, deadline):
    orthogonal, norms, mu = [], [], []
    for row in basis:
        check_time(deadline)
        vector = list(map(Fraction, row))
        coefficients = []
        for previous, norm in zip(orthogonal, norms):
            coefficient = dot(row, previous) / norm
            coefficients.append(coefficient)
            if coefficient:
                vector = [x - coefficient * y for x, y in zip(vector, previous)]
        orthogonal.append(vector)
        norms.append(dot(vector, vector))
        mu.append(coefficients)
    return orthogonal, norms, mu


def lll(basis, deadline):
    # 정수 행 기저. 크기 축소는 직교 벡터를 바꾸지 않는다.
    orthogonal, norms, mu = gram_schmidt(basis, deadline)
    k = 1
    while k < len(basis):
        check_time(deadline)
        for j in range(k - 1, -1, -1):
            q = round(mu[k][j])
            if q:
                basis[k] = [x - q * y for x, y in zip(basis[k], basis[j])]
                for i in range(j):
                    mu[k][i] -= q * mu[j][i]
                mu[k][j] -= q
        if norms[k] >= (Fraction(3, 4) - mu[k][k - 1] ** 2) * norms[k - 1]:
            k += 1
        else:
            basis[k - 1], basis[k] = basis[k], basis[k - 1]
            orthogonal, norms, mu = gram_schmidt(basis, deadline)
            k = max(1, k - 1)
    return basis, orthogonal, norms


def recovery_basis(n, deadline):
    # x_i - d_i ≡ A^i x_0 (mod 2^48), d_0 = 0, d_(i+1) = A d_i + C.
    # 첫 좌표가 x_0인 격자의 나머지 좌표는 MOD의 배수만큼 달라도 된다.
    basis = [[MOD if i == j else 0 for i in range(n)] for j in range(1, n)]
    basis.append([pow(A, i, MOD) for i in range(n)])
    return lll(basis, deadline)


def match_public_seed(curve, reduced, deadline):
    seed = FIPS[curve]
    basis, orthogonal, norms = reduced
    # 공개 바이트가 정하는 각 구간의 중점을 Babai 최근접 평면법에 넣는다.
    target = []
    offset = 0
    for byte in seed[:len(basis)]:
        target.append((byte << BYTE_SHIFT) + (1 << (BYTE_SHIFT - 1)) - offset)
        offset = step(offset)
    residual = target[:]
    for row, perpendicular, norm in reversed(list(zip(basis, orthogonal, norms))):
        check_time(deadline)
        q = round(dot(residual, perpendicular) / norm)
        residual = [x - q * y for x, y in zip(residual, row)]
    first = (target[0] - residual[0]) & MASK
    state = first
    for byte in seed:  # 기저에 사용하지 않은 바이트도 포함해 전부 검산한다.
        if state >> BYTE_SHIFT != byte:
            return None
        state = step(state)
    return first


def report_recovered_state(curve, first, attempts):
    distance, t = first_marker(first)
    row = dict(status="PASS", first_state=first, steps=distance,
               unix_time=t, dimensions=attempts, aligned_unix_time=None)
    print(f"{curve}  PASS  20바이트 일치 (격자 {attempts[-1]}차원)")
    print(f"  첫 상태  0x{first:012X}  역산 {distance:,}회")
    print(f"  첫 초기화 표식  {utc_string(t)}")
    # 후보의 첫 바이트는 1, 21, 41, ... 번째 호출에서만 나온다.
    # 표식 사이의 거리를 20으로 나눈 나머지는 다섯 표식마다 반복된다.
    if (distance - 1) % math.gcd(1 << 16, SEED_BYTES):
        print("  20회 호출 경계에 맞는 초기화 표식 없음")
        return row
    while (distance - 1) % SEED_BYTES:
        delta, t = first_marker(srand48(t))
        distance += delta
    index = (distance - 1) // SEED_BYTES + 1
    row.update(aligned_unix_time=t, aligned_steps=distance, index=index)
    print(f"  후보 경계에 맞는 가장 가까운 표식 (후보 {index:,})")
    print(f"  {utc_string(t)}")
    return row


def experiment6(quick=False, deadline=None):
    heading(6)
    if quick:
        print("축소 실행에서 생략")
        return finish("SKIP", reason="축소 실행", curves={})
    if deadline is None:
        deadline = time.monotonic() + 480
    started = time.monotonic()
    bases, curves = {}, {}
    for curve in ("P-256", "P-192", "P-384", "P-521"):
        attempts = []
        try:
            for n in (10, 12, 14):
                check_time(deadline)
                attempts.append(n)
                if n not in bases:
                    bases[n] = recovery_basis(n, deadline)
                first = match_public_seed(curve, bases[n], deadline)
                if first is not None:
                    break
            else:
                first = None
            if first is None:
                reason = "20바이트와 일치하는 상태를 찾지 못함"
                curves[curve] = dict(status="SKIP", reason=reason, dimensions=attempts)
            else:
                curves[curve] = report_recovered_state(curve, first, attempts)
                continue
        except TimeoutError as exc:
            reason = str(exc)
            curves[curve] = dict(status="SKIP", reason=reason, dimensions=attempts)
        print(f"{curve}  SKIP  {reason}")
        if curve == "P-256" or time.monotonic() >= deadline:
            break
    for curve in ("P-192", "P-384", "P-521"):
        if curve not in curves:
            reason = "P-256 복구 미완료" if curves["P-256"]["status"] == "SKIP" else "시간 한도"
            curves[curve] = dict(status="SKIP", reason=reason, dimensions=[])
            print(f"{curve}  SKIP  {reason}")
    print("초기화 표식은 역산 65,536회마다 다시 나타남")
    status = "PASS" if all(row["status"] == "PASS" for row in curves.values()) else "SKIP"
    return finish(status, curves=curves, elapsed_seconds=time.monotonic() - started)


lll_result = experiment6(QUICK)

### 초기화 시각과 후보 번호

순방향으로 시드가 일치한 행과, LLL로 전체 바이트가 일치하고 후보 경계까지 맞춘 행을 모읍니다. 초기화 표식은 반복되므로 LLL로 얻은 날짜는 가능한 초기화 시각입니다. 같은 곡선은 한 번만 표시합니다.

In [ ]:
from matplotlib import dates as mdates
from matplotlib.ticker import MaxNLocator

found = {row["curve"]: (row["unix_time"], row["index"]) for row in (p256, p224)
         if row["seed_match"]}
for curve, row in lll_result["curves"].items():
    if row["status"] == "PASS" and row.get("aligned_unix_time") is not None:
        found.setdefault(curve, (row["aligned_unix_time"], row["index"]))
curves = sorted(found, key=lambda curve: found[curve][0])
dates = [datetime.fromtimestamp(found[curve][0], timezone.utc) for curve in curves]
indices = [found[curve][1] for curve in curves]

In [ ]:
labels = {
    'ko': {
        'timeline': '공개 시드를 재현하는 초기화 시각',
        'date': '초기화 날짜 (UTC)',
        'candidate': '시드가 일치한 후보 번호',
        'index': '후보 번호 (1부터 시작)',
        'note': 'lrand48, 후보당 20바이트, 일치한 곡선 {n}개',
        'empty': '일치하는 곡선 없음',
    },
    'en': {
        'timeline': 'Initialization times reproducing public seeds',
        'date': 'Initialization date (UTC)',
        'candidate': 'Candidate index matching each seed',
        'index': 'Candidate index (one-based)',
        'note': 'lrand48, 20 bytes per candidate, {n} matching curves',
        'empty': 'No matching curves',
    },
}

In [ ]:
for lang in chart_languages:
    t = labels[lang]
    fig, ax = plt.subplots()
    ax.scatter(dates, range(len(curves)), color=CHART_COLORS[0])
    ax.set_yticks(range(len(curves)), labels=curves)
    for i, date in enumerate(dates):
        ax.annotate(date.strftime("%Y-%m-%d"), (date, i), xytext=(0, 8),
                    textcoords="offset points", ha="center", color=CHART_TEXT, fontsize=8)
    if curves:
        locator = mdates.AutoDateLocator(minticks=3, maxticks=6, tz=timezone.utc)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d", tz=timezone.utc))
        ax.margins(x=0.15, y=0.25)
    else:
        ax.text(0.5, 0.5, t["empty"], transform=ax.transAxes, ha="center")
    ax.set(title=t["timeline"], xlabel=t["date"])
    finish_chart(fig, f"{CPU_RUNTIME}, {t['note'].format(n=len(curves))}")

In [ ]:
for lang in chart_languages:
    t = labels[lang]
    fig, ax = plt.subplots()
    bars = ax.barh(curves, indices, color=CHART_COLORS[0], height=0.55)
    if curves:
        largest = max(indices)
        ax.bar_label(bars, labels=[f"{v:,}" if v == largest else "" for v in indices],
                     color=CHART_TEXT, padding=4)
        ax.margins(x=0.15)
    else:
        ax.text(0.5, 0.5, t["empty"], transform=ax.transAxes, ha="center")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set(title=t["candidate"], xlabel=t["index"])
    finish_chart(fig, f"{CPU_RUNTIME}, {t['note'].format(n=len(curves))}")